[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/visualizacion_datos/01_exploracion/exploracion_hallazgos.ipynb)

# Fase 1: Exploración y Hallazgos
## Detección de Deslizamientos de Tierra con Machine Learning
**Asignatura:** Visualización de Datos · 2026  
**Dataset:** Landslide4Sense — Sentinel-1/2 · ALOS DEM · 14 canales · 3799 muestras  
**Objetivo:** Descubrir qué modelos y qué señales del terreno son más discriminativas para detectar deslizamientos.

---

In [ ]:
# ── Setup: detectar entorno (Colab vs. local) y configurar rutas ──────────────
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists('Landslides_-Applied-ML-Course'):
        os.system('git clone https://github.com/apmontesp/Landslides_-Applied-ML-Course.git')
    DATA_DIR = 'Landslides_-Applied-ML-Course/visualizacion_datos/data'
    FIG_DIR  = 'Landslides_-Applied-ML-Course/visualizacion_datos/data/figures'
else:
    DATA_DIR = '../data'
    FIG_DIR  = '../data/figures'

os.makedirs(FIG_DIR, exist_ok=True)
print(f'Entorno: {"Colab" if IN_COLAB else "Local"}')
print(f'DATA_DIR: {DATA_DIR}')

In [ ]:
# ── Importaciones ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

# Estilo global: fondo blanco, cuadrícula gris tenue, sin marcos innecesarios
sns.set_theme(style='whitegrid')
plt.rcParams.update({
    'figure.facecolor'  : 'white',
    'axes.facecolor'    : 'white',
    'axes.edgecolor'    : '#D1D5DB',
    'axes.grid'         : True,
    'grid.color'        : '#E5E7EB',
    'grid.linewidth'    : 0.8,
    'font.size'         : 11,
    'axes.titlesize'    : 13,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 11,
    'figure.figsize'    : (10, 5),
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
})

# Paleta del proyecto
ROJO    = '#D62728'
VERDE   = '#2CA02C'
VERDE_C = '#86EFAC'   # verde claro (protocolo alternativo)
PURPURA = '#7C3AED'
PURPURA_C = '#C4B5FD' # púrpura claro
GRIS    = '#9CA3AF'
GRIS_C  = '#E5E7EB'

print('Librerías cargadas. Estilo: fondo blanco, cuadrícula gris tenue.')

In [ ]:
# ── Carga de datos de resultados ──────────────────────────────────────────────
import json as _json

_df_raw = pd.read_csv(f'{DATA_DIR}/comparison_table.csv')
for col in ['F1 medio', 'Std', 'AUC-ROC', 'Precisión', 'Recall', 'IoU']:
    _df_raw[col] = pd.to_numeric(_df_raw[col], errors='coerce')

df_channels = pd.read_csv(f'{DATA_DIR}/channel_stats_by_class.csv')

with open(f'{DATA_DIR}/final_summary.json') as f:
    summary = _json.load(f)

# df_models se sobreescribe en la Exploración 1 tras limpiar AUC; aquí solo preview
print('Datos cargados:')
print(f'  Modelos evaluados : {len(_df_raw)}')
print(f'  Canales analizados: {len(df_channels)}')
print(f'  Mejor modelo      : {summary["best_model"]} (F1={summary["best_f1"]:.4f})')
print()
_df_raw

## 1. Pregunta de Negocio

> **¿Qué modelo de Machine Learning detecta mejor los deslizamientos de tierra en imágenes satelitales multiespectrales, y qué características físicas del terreno son más determinantes para la predicción?**

**Contexto:** Los deslizamientos de tierra causan miles de muertes y miles de millones de dólares en pérdidas anuales. Detectarlos automáticamente desde imágenes satelitales permitiría alertas tempranas y mapeo rápido de riesgo. El dataset Landslide4Sense contiene imágenes de 14 bandas (ópticas, SAR, topográficas) etiquetadas como landslide (positivo) o no-landslide (negativo).

**Modelos evaluados:**
- **Clásicos:** Logistic Regression, SVM (RBF), Random Forest
- **Deep Learning:** ResNet-50, EfficientNet-B4, U-Net ResNet-34

**Métricas clave:** F1 Score (balance precisión-recall), AUC-ROC, Recall (prioridad en alertas tempranas)

---
## 2. Exploración 1: Que modelo tiene mayor F1 Score?

**Hipótesis inicial:** Esperamos que los modelos de Deep Learning superen a los clásicos, dado su mayor capacidad de representación.

Comenzamos con una visualización exploratoria directa: barras simples de F1 medio por modelo.

In [ ]:
# ── Exploración 1: F1 Score y AUC-ROC por modelo (barras agrupadas) ──────────
# Dos barras por modelo: F1 Score (métrica principal) y AUC-ROC (capacidad
# discriminativa global). AUC-ROC de modelos DL obtenido desde sus JSON de folds.

df = pd.read_csv(f'{DATA_DIR}/comparison_table.csv')

# Convertir columnas con '—' a numérico
for col in ['F1 medio', 'Std', 'AUC-ROC', 'Precisión', 'Recall', 'IoU']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Completar AUC-ROC para modelos DL desde sus JSON
import json as _json
auc_extra = {
    'ResNet-50'     : None,
    'EfficientNet-B4': None,
    'U-Net ResNet-34': None,
}
try:
    with open(f'{DATA_DIR}/folds/resnet50_folds.json') as f:
        auc_extra['ResNet-50'] = _json.load(f)['aggregate']['mean_auc_roc']
    with open(f'{DATA_DIR}/folds/unet_folds.json') as f:
        auc_extra['U-Net ResNet-34'] = _json.load(f)['aggregate']['mean_auc_roc']
except FileNotFoundError:
    pass

for modelo, auc in auc_extra.items():
    if auc is not None:
        df.loc[df['Modelo'] == modelo, 'AUC-ROC'] = auc

df_sorted = df.sort_values('F1 medio', ascending=False).reset_index(drop=True)

# ── Gráfica ──
x      = np.arange(len(df_sorted))
ancho  = 0.38

fig, ax = plt.subplots(figsize=(11, 5))
ax.set_facecolor('white')

colores_f1  = [VERDE  if t == 'Clásico' else PURPURA   for t in df_sorted['Tipo']]
colores_auc = [VERDE_C if t == 'Clásico' else PURPURA_C for t in df_sorted['Tipo']]

barras_f1  = ax.bar(x - ancho/2, df_sorted['F1 medio'],
                    width=ancho, color=colores_f1,  label='F1 Score')
barras_auc = ax.bar(x + ancho/2, df_sorted['AUC-ROC'],
                    width=ancho, color=colores_auc, label='AUC-ROC',
                    edgecolor='none', hatch='//', alpha=0.85)

# Valores encima de cada barra
for bar in barras_f1:
    h = bar.get_height()
    if not np.isnan(h):
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.008,
                f'{h:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

for bar in barras_auc:
    h = bar.get_height()
    if not np.isnan(h):
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.008,
                f'{h:.3f}', ha='center', va='bottom', fontsize=9, color='#4B5563')

ax.set_xticks(x)
ax.set_xticklabels(df_sorted['Modelo'], rotation=15, ha='right')
ax.set_ylabel('Puntuación (0 = peor · 1 = mejor)', fontsize=11)
ax.set_xlabel('Modelo de Machine Learning', fontsize=11)
ax.set_title('F1 Score y AUC-ROC por Modelo — Detección de Deslizamientos\n(validación cruzada 5 folds · barras oscuras = F1 · barras tramadas = AUC-ROC)')
ax.set_ylim(0, 1.05)
ax.axhline(0.8, color=GRIS, linewidth=1, linestyle='--', zorder=0)
ax.text(len(df_sorted)-0.5, 0.81, 'F1 = 0.80 (umbral operativo)', fontsize=8, color=GRIS)

leyenda = [
    mpatches.Patch(color=VERDE,    label='Clásico — F1 Score'),
    mpatches.Patch(color=VERDE_C,  label='Clásico — AUC-ROC', hatch='//'),
    mpatches.Patch(color=PURPURA,  label='Deep Learning — F1 Score'),
    mpatches.Patch(color=PURPURA_C,label='Deep Learning — AUC-ROC', hatch='//'),
]
ax.legend(handles=leyenda, fontsize=9, loc='upper right')
ax.yaxis.grid(True, color='#E5E7EB', linewidth=0.8)
ax.xaxis.grid(False)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/exploracion_1_f1_barras.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# Guardar df limpio para reutilizar en siguientes celdas
df_models = df_sorted.copy()
print('Figura guardada: exploracion_1_f1_barras.png')

### Análisis Exploración 1

**Observacion inesperada:** El modelo **Random Forest (F1=0.837)** supera a todos los modelos de Deep Learning, incluyendo ResNet-50 (F1=0.784) y EfficientNet-B4 (F1=0.755). U-Net ResNet-34 — diseñada específicamente para segmentacion semantica — obtiene el peor resultado (F1=0.444).

**¿Por qué?** Exploramos mas en la siguiente visualizacion...

In [ ]:
# ── Exploración 1b: Trade-off Precisión vs Recall ─────────────────────────────
# Solo modelos con ambas métricas disponibles (df_models ya tiene las columnas limpias)

df_pr = df_models.dropna(subset=['Precisión', 'Recall']).copy()

fig, ax = plt.subplots(figsize=(8, 6))
ax.set_facecolor('white')

colores_scatter = {'Clásico': VERDE, 'Deep Learning': PURPURA}

for tipo, grupo in df_pr.groupby('Tipo'):
    sc = ax.scatter(
        grupo['Precisión'], grupo['Recall'],
        c=colores_scatter[tipo],
        s=grupo['F1 medio'] * 400,
        label=tipo, alpha=0.85,
        edgecolors='white', linewidth=1.5, zorder=5
    )
    for _, fila in grupo.iterrows():
        ax.annotate(
            fila['Modelo'],
            (fila['Precisión'], fila['Recall']),
            textcoords='offset points', xytext=(10, 6),
            fontsize=9, color='#374151'
        )

# Curvas iso-F1
for f1_val in [0.75, 0.80, 0.85]:
    prec = np.linspace(0.55, 0.99, 400)
    rec  = (f1_val * prec) / (2 * prec - f1_val + 1e-9)
    mask = (rec > 0.5) & (rec <= 1.0)
    ax.plot(prec[mask], rec[mask], '--', color='#D1D5DB', linewidth=1, zorder=1)
    idx = np.argmin(np.abs(prec[mask] - 0.87))
    ax.text(prec[mask][idx], rec[mask][idx] + 0.01,
            f'F1={f1_val}', fontsize=8, color='#9CA3AF')

ax.set_xlabel('Precisión  (fracción de alertas correctas)', fontsize=11)
ax.set_ylabel('Recall  (fracción de deslizamientos detectados)', fontsize=11)
ax.set_title('Trade-off Precisión vs Recall — Modelos Clásicos\n(tamaño de burbuja proporcional al F1 Score)')
ax.legend(title='Tipo de modelo', fontsize=9)
ax.set_xlim(0.68, 0.88)
ax.set_ylim(0.74, 1.00)
ax.yaxis.grid(True, color='#E5E7EB', linewidth=0.8)
ax.xaxis.grid(True, color='#E5E7EB', linewidth=0.8)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/exploracion_prec_recall.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Modelos graficados: {list(df_pr["Modelo"])}')

---
## 3. Exploración 2: Que canales espectrales discriminan mejor los deslizamientos?

**¿Pregunta?:** Existe diferencia estadistica entre la senal de pixeles de deslizamiento (positivos) vs. no-deslizamiento (negativos) en cada canal?

Calculamos el **delta** (diferencia de medias) como proxy de poder discriminativo.

In [ ]:
# ── Exploración 2: Poder discriminativo por canal espectral ───────────────────
# Un color por magnitud: de gris tenue (poca diferencia) a rojo intenso (mayor).
# Eje X: diferencia absoluta de intensidad media entre clases (0-1).

df_ch = pd.read_csv(f'{DATA_DIR}/channel_stats_by_class.csv').copy()
df_ch['|Delta|'] = df_ch['Delta'].abs()
df_ch = df_ch.sort_values('|Delta|', ascending=True).reset_index(drop=True)

# Color por magnitud (escala continua de gris a rojo)
norm_vals = df_ch['|Delta|'] / df_ch['|Delta|'].max()
cmap      = plt.cm.get_cmap('YlOrRd')   # amarillo → naranja → rojo
bar_cols  = [cmap(0.15 + 0.82 * v) for v in norm_vals]  # evitar blanco puro

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_facecolor('white')

barras = ax.barh(df_ch['Nombre'], df_ch['|Delta|'],
                 color=bar_cols, edgecolor='white', linewidth=0.5)

# Valor al final de cada barra
for bar, val in zip(barras, df_ch['|Delta|']):
    ax.text(val + 0.008, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9, color='#374151')

# Anotación en el canal más discriminativo
top_idx = df_ch['|Delta|'].idxmax()
ax.annotate(
    '  Canal más discriminativo\n  (suelo expuesto en deslizamientos)',
    xy=(df_ch.loc[top_idx, '|Delta|'], top_idx),
    xytext=(df_ch['|Delta|'].max() * 0.55, top_idx - 1.2),
    arrowprops=dict(arrowstyle='->', color=ROJO, lw=1.5),
    fontsize=9, color=ROJO
)

ax.set_xlabel(
    'Diferencia absoluta de intensidad entre clases\n'
    '|Media(deslizamiento) − Media(no deslizamiento)|  (escala 0–1)',
    fontsize=10
)
ax.set_ylabel('Canal espectral', fontsize=11)
ax.set_title(
    'Poder Discriminativo por Canal Espectral\n'
    '(mayor valor = señal más diferente entre zona con y sin deslizamiento)'
)
ax.xaxis.grid(True, color='#E5E7EB', linewidth=0.8)
ax.yaxis.grid(False)

# Barra de color como leyenda de magnitud
sm = plt.cm.ScalarMappable(cmap='YlOrRd', norm=plt.Normalize(vmin=0, vmax=df_ch['|Delta|'].max()))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.01, shrink=0.6)
cbar.set_label('Diferencia de señal (mayor = más discriminativo)', fontsize=9)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/exploracion_2_canales_delta.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print('Top 5 canales más discriminativos:')
print(df_ch.sort_values('|Delta|', ascending=False)[['Nombre', '|Delta|']].head(5).to_string(index=False))

### Análisis Exploración 2

**Patrón claro:** Los canales **RedEdge3 (B7, |Delta|=0.807)** y **RedEdge2 (B6, |Delta|=0.563)** sobresalen notablemente sobre el resto. Estos canales de borde rojo de Sentinel-2 son altamente sensibles a la vegetacion y a suelos expuestos — exactamente lo que caracteriza una zona de deslizamiento fresco.

Los canales opticos convencionales (Azul, Verde, Rojo, NIR) muestran poca diferencia entre clases, lo que explica por que los modelos que priorizan esos canales tienen menor rendimiento.

In [ ]:
# ── Exploración 3: Intensidad media por clase — Top 6 canales ─────────────────
df_top6 = df_ch.sort_values('|Delta|', ascending=False).head(6).reset_index(drop=True)
x_pos   = np.arange(len(df_top6))
ancho   = 0.36

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_facecolor('white')

b1 = ax.bar(x_pos - ancho/2, df_top6['Media_Pos'], ancho,
            label='Deslizamiento (positivo)', color=ROJO, alpha=0.85,
            yerr=df_top6['Std_Pos'], capsize=4, error_kw={'linewidth': 1})
b2 = ax.bar(x_pos + ancho/2, df_top6['Media_Neg'], ancho,
            label='Sin deslizamiento (negativo)', color='#3B82F6', alpha=0.85,
            yerr=df_top6['Std_Neg'], capsize=4, error_kw={'linewidth': 1})

ax.set_xticks(x_pos)
ax.set_xticklabels(df_top6['Nombre'], rotation=15, ha='right')
ax.set_ylabel('Intensidad media normalizada  (rango 0–1)', fontsize=11)
ax.set_xlabel('Canal espectral (Top 6 con mayor diferencia entre clases)', fontsize=11)
ax.set_title(
    'Intensidad Media por Clase — Top 6 Canales Más Discriminativos\n'
    '(barras de error = desviación estándar dentro de cada clase)'
)
ax.legend(fontsize=10)
ax.yaxis.grid(True, color='#E5E7EB', linewidth=0.8)
ax.xaxis.grid(False)
ax.set_ylim(0, ax.get_ylim()[1] * 1.12)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/exploracion_3_medias_clase.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# ── Exploración 4: Ranking F1 y Recall con valores anotados ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Panel izquierdo: ranking por F1 Score ──
df_f1 = df_models.sort_values('F1 medio', ascending=True).reset_index(drop=True)
colores_rank = [ROJO if m == 'Random Forest' else
                (VERDE if t == 'Clásico' else PURPURA)
                for m, t in zip(df_f1['Modelo'], df_f1['Tipo'])]

barras_izq = axes[0].barh(df_f1['Modelo'], df_f1['F1 medio'],
                           color=colores_rank, alpha=0.85, edgecolor='white')
for bar, val in zip(barras_izq, df_f1['F1 medio']):
    axes[0].text(val + 0.008, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9, fontweight='bold')

axes[0].axvline(0.80, color=GRIS, linewidth=1.2, linestyle='--')
axes[0].text(0.802, 0.3, 'Umbral\noperativo\n(F1=0.80)', fontsize=8,
             color=GRIS, va='bottom')
axes[0].set_xlabel('F1 Score  (harmónica de Precisión y Recall; rango 0–1)', fontsize=10)
axes[0].set_title('Ranking por F1 Score\n(rojo = mejor modelo)')
axes[0].set_xlim(0, 1.0)
axes[0].xaxis.grid(True, color='#E5E7EB', linewidth=0.8)
axes[0].yaxis.grid(False)
axes[0].set_facecolor('white')

# ── Panel derecho: Recall por modelo ──
df_rec = df_models.dropna(subset=['Recall']).sort_values('Recall', ascending=True).reset_index(drop=True)
colores_rec = [ROJO if m == 'Random Forest' else GRIS for m in df_rec['Modelo']]

barras_der = axes[1].barh(df_rec['Modelo'], df_rec['Recall'],
                           color=colores_rec, alpha=0.85, edgecolor='white')
for bar, val in zip(barras_der, df_rec['Recall']):
    axes[1].text(val + 0.008, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9, fontweight='bold')

axes[1].axvline(0.90, color=ROJO, linewidth=1.2, linestyle='--', alpha=0.7)
axes[1].text(0.902, 0.3, 'Recall\n0.90\n(mín. alerta)', fontsize=8,
             color=ROJO, va='bottom')
axes[1].set_xlabel(
    'Recall  (fracción de deslizamientos reales que el modelo detecta; rango 0–1)',
    fontsize=10
)
axes[1].set_title('Recall por Modelo\n(crítico para alertas tempranas)')
axes[1].set_xlim(0, 1.05)   # eje desde 0 para no distorsionar proporciones
axes[1].xaxis.grid(True, color='#E5E7EB', linewidth=0.8)
axes[1].yaxis.grid(False)
axes[1].set_facecolor('white')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/exploracion_4_recall.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
## 4. Exploración 5: Variabilidad entre folds — consistencia del modelo

**¿Pregunta?:** Las métricas promedio ocultan variabilidad. Un modelo con F1=0.83 pero std=0.03 puede ser poco fiable operativamente. ¿Son los modelos estables entre particiones del dataset?

Cargamos los resultados por fold de cada modelo para comparar su distribucion real.

In [ ]:
# ── Exploración 5: Cargar resultados por fold de cada modelo ──────────────────
# Los JSONs de folds estan en visualizacion_datos/data/folds/ (incluidos en el repo)
import json as _json

FOLDS_DIR = f'{DATA_DIR}/folds'

def load_folds_classical(path):
    with open(path) as f:
        d = _json.load(f)
    return [fold['best_f1'] for fold in d['folds']]

def load_folds_dl(path, key='f1_pixel_thr05'):
    with open(path) as f:
        d = _json.load(f)
    return [fold[key] for fold in d['folds']]

fold_data = {}
try:
    fold_data['Logistic Reg.']  = load_folds_classical(f'{FOLDS_DIR}/logistic_regression_folds.json')
    fold_data['SVM (RBF)']      = load_folds_classical(f'{FOLDS_DIR}/svm_folds.json')
    fold_data['Random Forest']  = load_folds_classical(f'{FOLDS_DIR}/random_forest_folds.json')
    fold_data['ResNet-50']      = load_folds_dl(f'{FOLDS_DIR}/resnet50_folds.json', key='f1_thr05')
    fold_data['U-Net ResNet34'] = load_folds_dl(f'{FOLDS_DIR}/unet_folds.json', key='f1_pixel_thr05')
    # EfficientNet-B4: sin datos por fold, se omite
    print('Folds cargados OK:')
    for k, v in fold_data.items():
        print(f'  {k:<20} folds={len(v)}  mean={np.mean(v):.4f}  std={np.std(v):.4f}')
except FileNotFoundError as e:
    print(f'Archivo no encontrado: {e}')
    fold_data = {}


In [ ]:
# ── Exploración 5a: Boxplot de F1 por modelo (distribucion real 5 folds) ──────
if fold_data:
    order = sorted(fold_data.keys(), key=lambda k: np.median(fold_data[k]), reverse=True)
    data_ordered = [fold_data[k] for k in order]

    VERDE   = '#2ca02c'
    PURPURA = '#9467bd'
    dl_models = {'ResNet-50', 'U-Net ResNet34'}
    box_colors = [PURPURA if m in dl_models else VERDE for m in order]

    fig, ax = plt.subplots(figsize=(10, 5))
    bp = ax.boxplot(data_ordered, patch_artist=True, notch=False,
                    medianprops=dict(color='black', linewidth=2),
                    whiskerprops=dict(linewidth=1.5),
                    capprops=dict(linewidth=1.5))

    for patch, color in zip(bp['boxes'], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    for i, vals in enumerate(data_ordered, 1):
        ax.scatter([i]*len(vals), vals, color='black', s=30, zorder=5, alpha=0.7)

    ax.set_xticks(range(1, len(order)+1))
    ax.set_xticklabels(order, rotation=15, ha='right')
    ax.set_ylabel('F1 Score (por fold)')
    ax.set_title('Distribucion de F1 Score por Modelo — 5-fold CV\n(cada punto = un fold, caja = rango intercuartil)')
    ax.set_ylim(0.35, 1.0)
    ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5)
    ax.text(len(order) + 0.1, 0.803, 'F1 = 0.80', fontsize=9, color='gray', va='bottom')

    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=VERDE, label='Clasico'),
                       Patch(facecolor=PURPURA, label='Deep Learning')]
    ax.legend(handles=legend_elements, loc='lower right')

    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/exploracion_5a_boxplot_folds.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figura guardada: exploracion_5a_boxplot_folds.png')

### Análisis Exploración 5a — Boxplot

**Patrón de estabilidad:** Random Forest muestra la caja mas estrecha — sus 5 folds oscilan apenas 0.012 puntos F1 (0.824–0.848). En cambio, SVM (RBF) presenta la mayor dispersion (std=0.030), con el fold 1 cayendo hasta F1=0.753, por debajo del umbral operativo de 0.80.

**U-Net ResNet-34** presenta una distribucion compacta pero en un rango bajo (0.68–0.71), lo que indica que su bajo rendimiento es sistematico y no un artefacto de una mala particion.

In [ ]:
# ── Exploración 5b: Heatmap fold x modelo ─────────────────────────────────────
if fold_data:
    df_folds = pd.DataFrame(fold_data, index=[f'Fold {i}' for i in range(1, 6)])

    fig, ax = plt.subplots(figsize=(9, 4))
    sns.heatmap(df_folds.T, annot=True, fmt='.3f', cmap='RdYlGn',
                vmin=0.40, vmax=0.90, linewidths=0.5, linecolor='white',
                ax=ax, cbar_kws={'label': 'F1 Score'})

    ax.set_title('Heatmap F1 Score — Fold x Modelo\n(verde = mejor rendimiento, rojo = peor)')
    ax.set_xlabel('Particion (fold)')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/exploracion_5b_heatmap_folds.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figura guardada: exploracion_5b_heatmap_folds.png')

    fold_means = df_folds.mean(axis=1)
    worst_fold = fold_means.idxmin()
    print(f'\nFold con menor F1 promedio entre modelos: {worst_fold} (media={fold_means.min():.4f})')
    print('Medias por fold:')
    print(fold_means.round(4).to_string())

### Análisis Exploración 5b — Heatmap

**Pregunta clave:** Si hay un fold sistemáticamente mas difícil para todos los modelos, eso sugiere una particion de datos geográficamente sesgada — ese fold contiene imagenes con caracteristicas espectrales distintas al resto.

La columna con valores mas bajos (mas rojo) identifica la particion mas exigente del dataset. Este analisis complementa la evaluacion LORO (Leave-One-Region-Out) del dashboard interactivo, donde el sesgo geografico se hace aun mas evidente.

---
## 5. El Hallazgo

Despues del analisis exploratorio, se articulan dos hallazgos principales:

---

### Hallazgo 1: Random Forest supera a las redes neuronales profundas

**Anomalía encontrada:** Contra la intuicion dominante en computer vision, el modelo **Random Forest (F1=0.837)** supera a todos los modelos de Deep Learning. Esto se explica porque:
- El dataset Landslide4Sense (3799 muestras) es **relativamente pequeño** para entrenar redes profundas desde cero
- Las features engineered (14 canales ya pre-procesados) dan ventaja a los metodos de ensamble sobre las CNN que deben aprender representaciones
- U-Net (F1=0.444) sufre especialmente por su alta capacidad parametrica sin suficientes datos de entrenamiento

**Implicación de negocio:** Para sistemas de deteccion con datasets limitados y features bien definidas, los modelos clásicos de ML son la primera eleccion, no las arquitecturas profundas.

---

### Hallazgo 2: Los canales RedEdge (B6, B7) son la senal mas discriminativa

**Correlación descubierta:** Las bandas espectrales de **borde rojo** (RedEdge) de Sentinel-2 muestran una diferencia de medias (Delta) hasta **10 veces mayor** que los canales opticos convencionales. Esta senal corresponde fisicamente a la respuesta espectral del suelo desnudo expuesto durante un deslizamiento.

**Implicación de negocio:** Priorizar imagenes con bandas RedEdge disponibles (Sentinel-2 Nivel 2A) maximizara la precision de deteccion. El **DEM de elevacion (Delta=0.195)** y **SAR-VH (Delta=0.188)** complementan la senal optica.

---

**Estos hallazgos guiaran el diseno del Dashboard aclaratorio (Fase 2).**

In [ ]:
# ── Resumen cuantitativo del hallazgo ─────────────────────────────────────────
print('='*60)
print('RESUMEN DE HALLAZGOS — EXPLORACION')
print('='*60)
print()
print('Hallazgo 1: Ranking de modelos por F1 Score')
for _, row in df_models.sort_values('F1 medio', ascending=False).iterrows():
    marker = '>>' if row['Modelo'] == 'Random Forest' else '  '
    print(f'  {marker} {row["Modelo"]:<25} F1={row["F1 medio"]:.4f}  [{row["Tipo"]}]')

print()
print('Hallazgo 2: Top 5 canales mas discriminativos (|Delta media|)')
df_channels_sorted = df_channels.copy()
df_channels_sorted['|Delta|'] = df_channels_sorted['Delta'].abs()
for _, row in df_channels_sorted.sort_values('|Delta|', ascending=False).head(5).iterrows():
    print(f'  Canal {int(row["Canal"]):2d} — {row["Nombre"]:<25} |Delta|={row["|Delta|"]:.4f}')

print()
if fold_data:
    print('Hallazgo 3: Estabilidad entre folds (std F1)')
    for modelo, folds in sorted(fold_data.items(), key=lambda x: np.std(x[1])):
        marker = '>>' if modelo == 'Random Forest' else '  '
        print(f'  {marker} {modelo:<20} mean={np.mean(folds):.4f}  std={np.std(folds):.4f}  [{"ESTABLE" if np.std(folds) < 0.015 else "VARIABLE"}'  + ']')

print()
print('-> Estos hallazgos seran el centro del Dashboard aclaratorio.')
print('='*60)
